# 04 — Regenerate comparisons from saved results

Use this notebook to regenerate the edited plots and summary from an existing completed run without fitting models or selecting settings. It reads saved model files, existing test predictions/labels for visualization, and prior metric tables; it does not rerun test scoring. Point RUN_DIR to a completed run. Keep rf_common.py, feature_importance.py, comparison_summary.py and the HTML template beside this notebook.

In [ ]:
# Locate this add-on folder whether Jupyter starts here or in 02_notebooks.
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image

here = Path.cwd().resolve()
helper_dir = next((p for base in [here, *here.parents]
                   for p in [base, base / '02_notebooks']
                   if (p / 'rf_common.py').is_file()), None)
if helper_dir is None:
    raise FileNotFoundError('Keep rf_common.py beside these notebooks and start Jupyter in the extracted add-on.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
PACKAGE_ROOT = helper_dir.parent
import rf_common as rf
print('Add-on folder:', PACKAGE_ROOT)

In [ ]:
# Reuse a completed run. You may also assign RUN_DIR directly here.
OUTPUT_BASE = Path(os.environ.get('WALLET_RF_OUTPUT_BASE', str(PACKAGE_ROOT / '04_results' / 'random_forest'))).expanduser().resolve()
override = os.environ.get('WALLET_RF_RUN_DIR')
if override:
    RUN_DIR = Path(override).expanduser().resolve()
else:
    latest = OUTPUT_BASE / 'latest_run.json'
    if not latest.exists():
        raise FileNotFoundError('Set WALLET_RF_RUN_DIR or RUN_DIR to an existing result folder containing models/ and model_manifest.json.')
    RUN_DIR = Path(json.loads(latest.read_text())['run_dir']).expanduser().resolve()
print('Existing fitted-model run:', RUN_DIR)

In [ ]:
# Uses existing saved models and results; no fitting or parameter selection occurs.
figure_dir = rf.make_figures(RUN_DIR)
import pandas as pd
importance = pd.read_csv(RUN_DIR / 'variable_importance_all_models.csv')
display(importance.loc[importance.variant.eq('fixed24h_5'),
                       ['model', 'label', 'coefficient', 'importance', 'measure']])

In [ ]:
display(Image(filename=str(figure_dir / 'variable_importance_comparison.png')))
display(Image(filename=str(figure_dir / 'roc_calibration_four_models.png')))
print('Open the self-contained report:', figure_dir / 'RF_logistic_resultssummary.html')

**Interpretation:** logistic |β| is per one refit-cohort SD for continuous predictors; missingness indicators remain on the 0→1 scale and are listed separately. RF Gini scores sum to one across all columns and describe training split contributions. Correlated price terms share the same underlying information. The smaller-feature view omits price columns only from the display, not the fitted models, and does not renormalize scores. Importance is not significance or causality. The report retains the earlier grouped validation permutation check when its plot exists.